# Q1 - Knowledge Distillation on CIFAR-100 (Google Colab runner)

This notebook:
1. installs the libraries,
2. lets you **upload the .py files** of this folder,
3. downloads CIFAR-100,
4. runs every script in order,
5. zips the `output` folder so you can download it.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

## Step 1 - Install libraries

In [ ]:
# timm provides EfficientNet-Lite0; onnx / onnxruntime are used for export and CPU inference.
!pip install -q timm onnx onnxruntime

## Step 2 - Upload the python files
Run the cell and select **all .py files** from the Q1 folder
(config.py, data.py, models.py, distillation.py, train_utils.py, train_teacher.py,
train_baseline.py, train_distill.py, inference.py, export_onnx.py, onnx_inference.py,
benchmark.py, plot.py).

In [ ]:
import os
from google.colab import files

# Work inside a dedicated folder so all outputs end up in /content/q1/output
WORK_DIR = "/content/q1"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Opens a file picker; the selected files are saved in the current folder.
uploaded = files.upload()

# Check that nothing is missing.
required = ["config.py", "data.py", "models.py", "distillation.py", "train_utils.py",
            "train_teacher.py", "train_baseline.py", "train_distill.py", "inference.py",
            "export_onnx.py", "onnx_inference.py", "benchmark.py", "plot.py"]
missing = [f for f in required if not os.path.exists(f)]
print("Missing files:", missing if missing else "none - all files uploaded")

## Step 3 - Load the CIFAR-100 dataset

In [ ]:
import torchvision

# Downloads and extracts CIFAR-100 into /content/q1/data/cifar-100-python
for is_train in (True, False):
    torchvision.datasets.CIFAR100(root="data", train=is_train, download=True)
print("Dataset ready.")

## Step 4 - Train the ResNet-50 teacher
Saves `output/checkpoints/teacher_resnet50.pth` and the log `output/logs/teacher_resnet50.csv`.

In [ ]:
!python train_teacher.py

## Step 5 - Train the baseline students (no distillation)

In [ ]:
!python train_baseline.py

## Step 6 - Train the students with knowledge distillation

In [ ]:
!python train_distill.py

## Step 7 - Export all students to ONNX

In [ ]:
!python export_onnx.py

## Step 8 - Inference check (PyTorch and ONNX Runtime on the CPU)

In [ ]:
# Test-set accuracy of all 6 models with the unified PyTorch inference pipeline
!python inference.py --model all --variant both

In [ ]:
# Test-set accuracy of all 6 models with ONNX Runtime (CPU)
!python onnx_inference.py --model all --variant both

## Step 9 - Benchmark (writes `output/benchmark.csv`)

In [ ]:
!python benchmark.py
import pandas as pd
pd.read_csv("output/benchmark.csv")

## Step 10 - Plots (one per recorded metric)

In [ ]:
!python plot.py
from IPython.display import Image, display
display(Image("output/plots/train_loss.png"))
display(Image("output/plots/test_accuracy.png"))

## Step 11 - Download everything in `output/`

In [ ]:
# Zips the whole output folder and downloads it to your computer.
!zip -r -q q1_output.zip output
files.download("q1_output.zip")